In [1]:
# Imports and configuration.
#
# This notebook runs an iterative greedy two-edge search:
#
#   1. Evaluate every unordered pair of host edges.
#   2. Select the pair with the greatest exact score reward.
#   3. Apply that pair even when its reward is zero or negative.
#   4. Refeed the altered graph through the same exhaustive pair analysis.
#   5. Repeat until MAX_ITERATIONS is reached or a previously visited
#      coloring is encountered.
#
# Progress messages are printed during the expensive pair scans.
# All detailed reporting is deferred to the final cell.

from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from ramsey import (
    RGraph,
    RProblem,
    RSQLiteArchive,
    RSearchState,
)
from ramsey.RAction import analyze_actions
from ramsey.RTwoEdgeFlipCausalAnalysis import (
    analyze_two_edge_flip_causality,
)

N_VERTICES = 43
MINIMUM_SCORE = 0
MAXIMUM_SCORE = 299
ARCHIVE_INDEX = 0
ARCHIVE_LIMIT = ARCHIVE_INDEX + 1

MAX_ITERATIONS = 10
PROGRESS_EDGE_INTERVAL = 50
TOP_PAIR_COUNT = 25

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

In [2]:
# Load one archived sub-300 coloring.
#
# Change ARCHIVE_INDEX in the configuration cell to select a different
# record from the archive query result.

graph = RGraph(
    RProblem.r55(
        n_vertices=N_VERTICES,
    )
)

archive = RSQLiteArchive(DATABASE_PATH)

records = archive.colorings_in_score_range(
    minimum_score=MINIMUM_SCORE,
    maximum_score=MAXIMUM_SCORE,
    limit=ARCHIVE_LIMIT,
    graph=graph,
)

if len(records) <= ARCHIVE_INDEX:
    raise RuntimeError(
        f"Archive returned only {len(records)} records; "
        f"cannot select index {ARCHIVE_INDEX}."
    )

record = records[ARCHIVE_INDEX]
archived = archive.load_coloring(
    record.coloring_id,
    graph,
)
state = RSearchState(
    archived.coloring
)

if state.score != record.score:
    raise RuntimeError(
        "The loaded state score does not match the archive record."
    )

initial_state = state.copy()
initial_colors = state.colors.copy()
initial_score = int(state.score)

In [3]:
# Helper functions for hashing states and exhaustively scoring all pairs.

def coloring_key(search_state: RSearchState) -> bytes:
    """Return an exact immutable key for the current edge coloring."""
    return np.packbits(
        search_state.colors,
        bitorder="little",
    ).tobytes()


def scan_all_two_edge_pairs(
    search_state: RSearchState,
    *,
    iteration: int,
) -> dict:
    """
    Exhaustively evaluate every unordered pair of edge flips.

    The supplied state is not mutated. Progress output is printed while
    the scan is running. The returned dictionary contains the complete
    pair arrays and the best-pair summary.
    """
    number_of_edges = search_state.number_of_edges
    number_of_pairs = (
        number_of_edges * (number_of_edges - 1) // 2
    )

    single_rewards = analyze_actions(
        search_state
    ).immediate_rewards.astype(
        np.int32,
        copy=True,
    )

    first_edges = np.empty(
        number_of_pairs,
        dtype=np.uint16,
    )
    second_edges = np.empty(
        number_of_pairs,
        dtype=np.uint16,
    )
    pair_rewards = np.empty(
        number_of_pairs,
        dtype=np.int32,
    )

    pair_index = 0
    started = perf_counter()

    print(
        f"Iteration {iteration:02d}: "
        f"scanning {number_of_pairs:,} pairs "
        f"from score {search_state.score}"
    )

    for first_edge in range(number_of_edges - 1):
        working_state = search_state.copy()
        working_state.apply_edge_flip(first_edge)

        for second_edge in range(
            first_edge + 1,
            number_of_edges,
        ):
            working_state.apply_edge_flip(second_edge)

            first_edges[pair_index] = first_edge
            second_edges[pair_index] = second_edge
            pair_rewards[pair_index] = (
                search_state.score
                - working_state.score
            )
            pair_index += 1

            working_state.apply_edge_flip(second_edge)

        if (
            first_edge % PROGRESS_EDGE_INTERVAL == 0
            or first_edge == number_of_edges - 2
        ):
            elapsed = perf_counter() - started
            print(
                f"  first edge "
                f"{first_edge:3d}/{number_of_edges - 2} | "
                f"pairs {pair_index:,}/{number_of_pairs:,} | "
                f"elapsed {elapsed:,.1f}s"
            )

    elapsed = perf_counter() - started

    if pair_index != number_of_pairs:
        raise RuntimeError(
            "The pair scan did not produce the expected pair count."
        )

    best_index = int(np.argmax(pair_rewards))
    best_first_edge = int(first_edges[best_index])
    best_second_edge = int(second_edges[best_index])
    best_pair_reward = int(pair_rewards[best_index])

    individual_sums = (
        single_rewards[first_edges]
        + single_rewards[second_edges]
    )
    interaction_rewards = (
        pair_rewards - individual_sums
    )

    print(
        f"Iteration {iteration:02d}: "
        f"best pair "
        f"({best_first_edge}, {best_second_edge}), "
        f"reward {best_pair_reward:+d}, "
        f"scan {elapsed:,.3f}s"
    )

    return {
        "single_rewards": single_rewards,
        "first_edges": first_edges,
        "second_edges": second_edges,
        "pair_rewards": pair_rewards,
        "interaction_rewards": interaction_rewards,
        "best_index": best_index,
        "best_first_edge": best_first_edge,
        "best_second_edge": best_second_edge,
        "best_pair_reward": best_pair_reward,
        "elapsed": float(elapsed),
        "number_of_pairs": int(number_of_pairs),
    }

In [4]:
# Run the iterative greedy two-edge search.
#
# At every iteration, the globally best two-edge move is applied,
# regardless of whether its reward is positive, zero, or negative.
#
# A repeated coloring stops the run because continuing would enter a cycle.
# MAX_ITERATIONS is the other stopping condition.

iteration_records = []
visited_states = {
    coloring_key(state): 0
}
termination_reason = None
search_started = perf_counter()

for iteration in range(1, MAX_ITERATIONS + 1):
    scan = scan_all_two_edge_pairs(
        state,
        iteration=iteration,
    )

    best_first_edge = scan["best_first_edge"]
    best_second_edge = scan["best_second_edge"]
    best_pair_reward = scan["best_pair_reward"]

    score_before = int(state.score)

    causal = analyze_two_edge_flip_causality(
        state,
        best_first_edge,
        best_second_edge,
    )

    if causal.exact_reward != best_pair_reward:
        raise RuntimeError(
            "The causal-analysis reward does not match "
            "the exhaustive pair scan."
        )

    first_sequential_reward = int(
        state.apply_edge_flip(best_first_edge)
    )
    second_sequential_reward = int(
        state.apply_edge_flip(best_second_edge)
    )

    score_after = int(state.score)
    actual_pair_reward = (
        score_before - score_after
    )

    if actual_pair_reward != best_pair_reward:
        raise RuntimeError(
            "The applied pair reward does not match "
            "the exhaustive pair scan."
        )

    destroyed_red = sum(
        change.destroyed and change.color == 0
        for change in causal.clique_changes
    )
    destroyed_blue = sum(
        change.destroyed and change.color == 1
        for change in causal.clique_changes
    )
    created_red = sum(
        change.created and change.color == 0
        for change in causal.clique_changes
    )
    created_blue = sum(
        change.created and change.color == 1
        for change in causal.clique_changes
    )

    pair_rewards = scan["pair_rewards"]
    interaction_rewards = scan["interaction_rewards"]

    iteration_records.append(
        {
            "iteration": int(iteration),
            "score_before": score_before,
            "score_after": score_after,
            "best_first_edge": best_first_edge,
            "best_second_edge": best_second_edge,
            "first_endpoints": tuple(
                int(vertex)
                for vertex in graph.edges[best_first_edge]
            ),
            "second_endpoints": tuple(
                int(vertex)
                for vertex in graph.edges[best_second_edge]
            ),
            "first_individual_reward": int(
                scan["single_rewards"][best_first_edge]
            ),
            "second_individual_reward": int(
                scan["single_rewards"][best_second_edge]
            ),
            "first_sequential_reward": first_sequential_reward,
            "second_sequential_reward": second_sequential_reward,
            "pair_reward": best_pair_reward,
            "interaction_reward": int(
                interaction_rewards[scan["best_index"]]
            ),
            "best_single_reward": int(
                scan["single_rewards"].max()
            ),
            "improving_pairs": int(
                np.count_nonzero(pair_rewards > 0)
            ),
            "neutral_pairs": int(
                np.count_nonzero(pair_rewards == 0)
            ),
            "worsening_pairs": int(
                np.count_nonzero(pair_rewards < 0)
            ),
            "minimum_pair_reward": int(
                pair_rewards.min()
            ),
            "maximum_pair_reward": int(
                pair_rewards.max()
            ),
            "mean_pair_reward": float(
                pair_rewards.mean()
            ),
            "std_pair_reward": float(
                pair_rewards.std()
            ),
            "destroyed_red": int(destroyed_red),
            "destroyed_blue": int(destroyed_blue),
            "created_red": int(created_red),
            "created_blue": int(created_blue),
            "changed_vertices": int(
                len(causal.changed_vertices)
            ),
            "changed_structure_edges": int(
                len(causal.changed_structure_edges)
            ),
            "changed_future_rewards": int(
                np.count_nonzero(
                    causal.greedy_reward_delta
                )
            ),
            "shared_vertex": bool(
                causal.shared_vertex
            ),
            "scan_seconds": float(
                scan["elapsed"]
            ),
        }
    )

    print(
        f"Iteration {iteration:02d}: "
        f"applied "
        f"({best_first_edge}, {best_second_edge}), "
        f"score {score_before} -> {score_after}, "
        f"reward {actual_pair_reward:+d}"
    )

    new_key = coloring_key(state)

    if new_key in visited_states:
        first_seen = visited_states[new_key]
        termination_reason = (
            f"Repeated coloring detected after iteration "
            f"{iteration}; first seen at iteration {first_seen}."
        )
        print(termination_reason)
        break

    visited_states[new_key] = iteration

else:
    termination_reason = (
        f"Reached MAX_ITERATIONS={MAX_ITERATIONS}."
    )

total_search_seconds = (
    perf_counter() - search_started
)

Iteration 01: scanning 407,253 pairs from score 129
  first edge   0/901 | pairs 902/407,253 | elapsed 0.3s
  first edge  50/901 | pairs 44,727/407,253 | elapsed 11.9s
  first edge 100/901 | pairs 86,052/407,253 | elapsed 23.0s
  first edge 150/901 | pairs 124,877/407,253 | elapsed 33.3s
  first edge 200/901 | pairs 161,202/407,253 | elapsed 43.1s
  first edge 250/901 | pairs 195,027/407,253 | elapsed 52.5s
  first edge 300/901 | pairs 226,352/407,253 | elapsed 61.2s
  first edge 350/901 | pairs 255,177/407,253 | elapsed 69.5s
  first edge 400/901 | pairs 281,502/407,253 | elapsed 77.1s
  first edge 450/901 | pairs 305,327/407,253 | elapsed 84.2s
  first edge 500/901 | pairs 326,652/407,253 | elapsed 90.9s
  first edge 550/901 | pairs 345,477/407,253 | elapsed 97.5s
  first edge 600/901 | pairs 361,802/407,253 | elapsed 103.2s
  first edge 650/901 | pairs 375,627/407,253 | elapsed 108.4s
  first edge 700/901 | pairs 386,952/407,253 | elapsed 113.1s
  first edge 750/901 | pairs 395,777/

In [5]:
# Build final tables and summary objects without printing them yet.

history = pd.DataFrame(
    iteration_records
)

if history.empty:
    raise RuntimeError(
        "The search produced no iteration records."
    )

best_score = int(
    min(
        initial_score,
        int(history["score_after"].min()),
    )
)
best_iteration_rows = history[
    history["score_after"] == best_score
]
best_iteration = (
    0
    if best_score == initial_score
    else int(best_iteration_rows.iloc[0]["iteration"])
)

final_single_rewards = analyze_actions(
    state
).immediate_rewards.astype(
    np.int32,
    copy=True,
)

score_path = [initial_score] + [
    int(value)
    for value in history["score_after"]
]

summary = {
    "archive_id": int(record.coloring_id),
    "archived_score": int(record.score),
    "initial_score": initial_score,
    "final_score": int(state.score),
    "best_score": best_score,
    "best_iteration": best_iteration,
    "iterations_completed": int(len(history)),
    "distinct_states_seen": int(len(visited_states)),
    "termination_reason": termination_reason,
    "total_search_seconds": float(
        total_search_seconds
    ),
    "final_best_single_reward": int(
        final_single_rewards.max()
    ),
}

In [6]:
# Final consolidated report.
#
# This is the only detailed output cell. All integer-valued quantities
# are explicitly formatted as integers.

separator = "=" * 92
subseparator = "-" * 92

print(separator)
print("GREEDY TWO-EDGE FLIP SEARCH REPORT")
print(separator)
print()

print(subseparator)
print("ARCHIVE AND CONFIGURATION")
print(subseparator)
print(f"Archive ID                  : {summary['archive_id']}")
print(f"Archived Score              : {summary['archived_score']}")
print(f"Initial Loaded Score        : {summary['initial_score']}")
print(f"Number of Vertices          : {N_VERTICES}")
print(f"Number of Edges             : {state.number_of_edges:,}")
print(f"Pairs per Iteration         : {history.iloc[0]['improving_pairs'] + history.iloc[0]['neutral_pairs'] + history.iloc[0]['worsening_pairs']:,}")
print(f"Maximum Iterations          : {MAX_ITERATIONS}")
print()

print(subseparator)
print("SEARCH RESULT")
print(subseparator)
print(f"Iterations Completed        : {summary['iterations_completed']}")
print(f"Distinct States Seen        : {summary['distinct_states_seen']}")
print(f"Initial Score               : {summary['initial_score']}")
print(f"Final Score                 : {summary['final_score']}")
print(f"Best Score Reached          : {summary['best_score']}")
print(f"Best Score First Iteration  : {summary['best_iteration']}")
print(f"Final Best Single Reward    : {summary['final_best_single_reward']:+d}")
print(f"Total Search Time (s)       : {summary['total_search_seconds']:.3f}")
print(f"Termination                 : {summary['termination_reason']}")
print()

print(subseparator)
print("SCORE PATH")
print(subseparator)
print(
    " -> ".join(
        str(int(score))
        for score in score_path
    )
)
print()

print(subseparator)
print("ITERATION HISTORY")
print(subseparator)

header = (
    f"{'iter':>4} "
    f"{'score':>11} "
    f"{'pair':>13} "
    f"{'reward':>7} "
    f"{'interact':>9} "
    f"{'best1':>6} "
    f"{'improve':>8} "
    f"{'neutral':>8} "
    f"{'worsen':>8} "
    f"{'scan(s)':>9}"
)
print(header)
print("-" * len(header))

for row in iteration_records:
    pair_text = (
        f"({row['best_first_edge']},"
        f"{row['best_second_edge']})"
    )
    score_text = (
        f"{row['score_before']}"
        f"->{row['score_after']}"
    )

    print(
        f"{row['iteration']:4d} "
        f"{score_text:>11} "
        f"{pair_text:>13} "
        f"{row['pair_reward']:+7d} "
        f"{row['interaction_reward']:+9d} "
        f"{row['best_single_reward']:+6d} "
        f"{row['improving_pairs']:8,d} "
        f"{row['neutral_pairs']:8,d} "
        f"{row['worsening_pairs']:8,d} "
        f"{row['scan_seconds']:9.3f}"
    )

print()

print(subseparator)
print("PER-ITERATION CAUSAL DETAILS")
print(subseparator)

for row in iteration_records:
    print(
        f"Iteration {row['iteration']:02d} | "
        f"score {row['score_before']} -> {row['score_after']} | "
        f"pair ({row['best_first_edge']}, {row['best_second_edge']})"
    )
    print(
        f"  Endpoints                  : "
        f"{row['first_endpoints']} and "
        f"{row['second_endpoints']}"
    )
    print(
        f"  Individual Rewards         : "
        f"{row['first_individual_reward']:+d}, "
        f"{row['second_individual_reward']:+d}"
    )
    print(
        f"  Sequential Rewards         : "
        f"{row['first_sequential_reward']:+d}, "
        f"{row['second_sequential_reward']:+d}"
    )
    print(
        f"  Exact Pair Reward          : "
        f"{row['pair_reward']:+d}"
    )
    print(
        f"  Interaction Reward         : "
        f"{row['interaction_reward']:+d}"
    )
    print(
        f"  Pair Reward Range          : "
        f"{row['minimum_pair_reward']:+d} to "
        f"{row['maximum_pair_reward']:+d}"
    )
    print(
        f"  Pair Reward Mean / Std     : "
        f"{row['mean_pair_reward']:.3f} / "
        f"{row['std_pair_reward']:.3f}"
    )
    print(
        f"  Destroyed K5s (R/B)        : "
        f"{row['destroyed_red']}/"
        f"{row['destroyed_blue']}"
    )
    print(
        f"  Created K5s (R/B)          : "
        f"{row['created_red']}/"
        f"{row['created_blue']}"
    )
    print(
        f"  Changed Vertices           : "
        f"{row['changed_vertices']}"
    )
    print(
        f"  Changed Structural Edges   : "
        f"{row['changed_structure_edges']}"
    )
    print(
        f"  Changed Future Rewards     : "
        f"{row['changed_future_rewards']}"
    )
    print(
        f"  Shared Vertex              : "
        f"{row['shared_vertex']}"
    )
    print()

print(separator)
print("END OF REPORT")
print(separator)

GREEDY TWO-EDGE FLIP SEARCH REPORT

--------------------------------------------------------------------------------------------
ARCHIVE AND CONFIGURATION
--------------------------------------------------------------------------------------------
Archive ID                  : 3127
Archived Score              : 129
Initial Loaded Score        : 129
Number of Vertices          : 43
Number of Edges             : 903
Pairs per Iteration         : 407,253
Maximum Iterations          : 10

--------------------------------------------------------------------------------------------
SEARCH RESULT
--------------------------------------------------------------------------------------------
Iterations Completed        : 2
Distinct States Seen        : 2
Initial Score               : 129
Final Score                 : 129
Best Score Reached          : 129
Best Score First Iteration  : 0
Final Best Single Reward    : +0
Total Search Time (s)       : 257.231
Termination                 : Repeated co